In [1]:
import numpy as np
import pandas as pd


# =========================================================
# Configuration
# =========================================================

FS = 1.0
WINDOW_LENGTH = 60

# Variables for which frequency-domain features are meaningful
frequency_variables = [
    # Temperatures
    "T701",
    "T702",
    "T703",
    "T704",
    "T705",
    "T706",
    "T708",
    "T709",
    "T711",
    "T712",

    # Pressure
    "P701",
    "P702",
    "PDI701",
    "PDI702",
    "PY23",

    # Flow
    "FT703",
    "FT704",
    "FYI702",

    # Level
    "LS701",
    "LS702",
]

group_columns = [
    "identifier",
    "window",
]


# =========================================================
# Load data
# =========================================================

operation_data = pd.read_parquet(
    "../data/processed/operation_data.parquet"
)

temporal_features = pd.read_parquet(
    "../data/processed/temporal_features.parquet"
)


# =========================================================
# Frequency-domain feature calculation
# =========================================================

def calculate_frequency_features(signal):
    """
    Calculate frequency-domain features for one complete
    60-second signal sampled at 1 Hz.
    """

    signal = np.asarray(signal, dtype=float)

    # Only complete, valid windows
    if len(signal) != WINDOW_LENGTH or np.isnan(signal).any():
        return {
            "total_power": np.nan,
            "dominant_power_ratio": np.nan,
            "spectral_entropy": np.nan,
        }

    # Remove DC component
    signal = signal - np.mean(signal)

    # FFT
    fft_values = np.fft.rfft(signal)

    frequencies = np.fft.rfftfreq(
        WINDOW_LENGTH,
        d=1 / FS,
    )

    # Power spectrum
    power = np.abs(fft_values) ** 2 / WINDOW_LENGTH

    # Remove DC component
    frequencies = frequencies[1:]
    power = power[1:]

    # Total spectral power
    total_power = np.sum(power)

    # Constant signal
    if total_power == 0:
        return {
            "total_power": 0.0,
            "dominant_power_ratio": np.nan,
            "spectral_entropy": 0.0,
        }

    # Dominant frequency component
    dominant_idx = np.argmax(power)
    dominant_power = power[dominant_idx]

    dominant_power_ratio = (
        dominant_power / total_power
    )

    # Spectral entropy
    power_distribution = power / total_power

    spectral_entropy = -np.sum(
        power_distribution
        * np.log2(power_distribution + 1e-12)
    )

    return {
        "total_power": total_power,
        "dominant_power_ratio": dominant_power_ratio,
        "spectral_entropy": spectral_entropy,
    }


# =========================================================
# Create frequency features
# =========================================================

frequency_feature_list = []

for variable in frequency_variables:

    grouped = operation_data.groupby(
        group_columns,
        sort=False,
    )[variable]

    for (identifier, window), signal in grouped:

        features = calculate_frequency_features(
            signal.to_numpy()
        )

        frequency_feature_list.append({
            "identifier": identifier,
            "window": window,
            "variable": variable,
            **features,
        })


frequency_features_all = pd.DataFrame(
    frequency_feature_list
)


# =========================================================
# Pivot frequency features
# One row = one experiment window
# =========================================================

frequency_features_ml = (
    frequency_features_all
    .pivot(
        index=group_columns,
        columns="variable",
        values=[
            "total_power",
            "dominant_power_ratio",
            "spectral_entropy",
        ],
    )
)


# Flatten MultiIndex columns
frequency_features_ml.columns = [
    f"{feature}_{variable}"
    for feature, variable in frequency_features_ml.columns
]

frequency_features_ml = (
    frequency_features_ml
    .reset_index()
)


# =========================================================
# Prepare temporal features
# =========================================================

metadata_columns = [
    "identifier",
    "window",
    "window_start",
    "window_end",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "anomaly_label",
]

temporal_feature_columns = [
    column
    for column in temporal_features.columns
    if column not in metadata_columns
]

temporal_features_ml = temporal_features[
    group_columns + temporal_feature_columns
].copy()


# =========================================================
# Merge temporal + frequency features
# =========================================================

ml_features = temporal_features_ml.merge(
    frequency_features_ml,
    on=group_columns,
    how="inner",
    validate="one_to_one",
)


# =========================================================
# Final metadata / target
# =========================================================

metadata = temporal_features[
    [
        "identifier",
        "window",
        "batch",
        "operating_point",
        "experiment_type",
        "experiment",
        "anomaly_label",
    ]
].copy()

metadata = metadata.drop_duplicates(
    subset=["identifier", "window"]
)


ml_features = ml_features.merge(
    metadata,
    on=["identifier", "window"],
    how="left",
    validate="one_to_one",
)


# =========================================================
# Arrange columns
# =========================================================

metadata_order = [
    "identifier",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "window",
    "anomaly_label",
]

feature_columns = [
    column
    for column in ml_features.columns
    if column not in metadata_order
]

ml_features = ml_features[
    metadata_order + feature_columns
]


# =========================================================
# Save to parquet
# =========================================================

ml_features.to_parquet(
    "../data/processed/notebooks/ml_features.parquet",
    index=False
)

print("Data saved to parquet!")


# =========================================================
# Final validation
# =========================================================

model_feature_columns = [
    column
    for column in ml_features.columns
    if column not in metadata_order
]

frequency_columns = [
    column
    for column in model_feature_columns
    if (
        "total_power" in column
        or "dominant_power_ratio" in column
        or "spectral_entropy" in column
    )
]


print("\nFinal ML dataset")
print("----------------")
print("Shape:", ml_features.shape)
print("ML features:", len(model_feature_columns))
print("Frequency features:", len(frequency_columns))
print("Frequency variables:", len(frequency_variables))

print(
    "Expected frequency features:",
    len(frequency_variables) * 3,
)

print(
    "Duplicate windows:",
    ml_features.duplicated(
        subset=["identifier", "window"]
    ).sum(),
)

print(
    "Missing target:",
    ml_features["anomaly_label"].isna().sum(),
)

print("\nMissing metadata:")

print(
    ml_features[
        [
            "batch",
            "operating_point",
            "experiment_type",
            "experiment",
        ]
    ]
    .isna()
    .sum()
)

print("\nTarget distribution:")

print(
    ml_features["anomaly_label"]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing feature values:",
    ml_features[model_feature_columns]
    .isna()
    .sum()
    .sum(),
)

print("\nFrequency feature columns:")
for column in frequency_columns:
    print(column)

print("\nFinal column order:")
print(ml_features.columns.tolist())

Data saved to parquet!

Final ML dataset
----------------
Shape: (11512, 787)
ML features: 780
Frequency features: 60
Frequency variables: 20
Expected frequency features: 60
Duplicate windows: 0
Missing target: 0

Missing metadata:
batch              0
operating_point    0
experiment_type    0
experiment         0
dtype: int64

Target distribution:
anomaly_label
0.0    8887
1.0     386
2.0    1698
3.0     541
Name: count, dtype: int64

Missing feature values: 191058

Frequency feature columns:
total_power_FT703
total_power_FT704
total_power_FYI702
total_power_LS701
total_power_LS702
total_power_P701
total_power_P702
total_power_PDI701
total_power_PDI702
total_power_PY23
total_power_T701
total_power_T702
total_power_T703
total_power_T704
total_power_T705
total_power_T706
total_power_T708
total_power_T709
total_power_T711
total_power_T712
dominant_power_ratio_FT703
dominant_power_ratio_FT704
dominant_power_ratio_FYI702
dominant_power_ratio_LS701
dominant_power_ratio_LS702
dominant_power_